In [0]:
%run "/Workspace/Users/ashish.bv.singh@accenture.com/__init__"

In [0]:
df = spark.read.csv("/Volumes/scd/source/sourcefiles/", header=True, inferSchema=True)

In [0]:
# This code is for legacy  without autoloader
type1_table_name = "scd.type1.target"
if not spark.catalog.tableExists(type1_table_name):
    print(f"Performing full load for {type1_table_name}")
    df.write.format("delta").saveAsTable(f"{type1_table_name}")
else: 
    print(f"Performing SCD1 load for {type1_table_name}")
    target_delta_table = DeltaTable.forName(spark, type1_table_name)
    target_delta_table.alias('t').merge(
        df.alias('s'),
        "t.customer_id = s.customer_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
    